# Lekcija 16 - Implementacija skalabilnih agenata s Microsoft Foundry

U ovom bilježniku gradite **agent za korisničku podršku spreman za proizvodnju** za fiktivnu tvrtku **Contoso**. Za razliku od ranijih lekcija, poanta ovdje nije petlja rezoniranja agenta — već sve ono *oko* nje što čini agenta sigurnim za izvođenje u velikom opsegu:

1. **Pozivanje alata** — pretraživanje narudžbi i kreiranje tiketa.
2. **RAG** — odgovori na osnovi politika iz baze znanja.
3. **Memorija** — pamćenje korisnika tijekom interakcija.
4. **Usmjeravanje modela** — slanje jednostavnih upita malom modelu, složenih velikom modelu.
5. **Keširanje odgovora** — posluživanje ponovljenih pitanja bez poziva modela.
6. **Ljudsko odobrenje** — povrati iznad praga se zaustavljaju radi potvrde.
7. **Vrata evaluacije** — offline test skup koji blokira lošu verziju.
8. **Promatranje** — OpenTelemetry praćenje oko svakog zahtjeva.

Svaki odjeljak je samostalan i pokretljiv. Pročitajte svaki redak — proizvodne primitives držimo namjerno malim.


## Postavljanje

Prije pokretanja ove bilježnice, uvjerite se da imate:

1. **Microsoft Foundry projekt** s implementiranim chat modelom (npr. `gpt-4.1-mini`).
2. **Prijavljeni ste pomoću Azure CLI** — pokrenite `az login` u vašem terminalu.
3. **Postavljene potrebne varijable okoline:**
   - `AZURE_AI_PROJECT_ENDPOINT` — endpoint vašeg Microsoft Foundry projekta.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — naziv vašeg implementiranog modela.

RAG dio koristi **Azure AI Search** kada su postavljeni `AZURE_SEARCH_SERVICE_ENDPOINT` i `AZURE_SEARCH_API_KEY`, a u suprotnom koristi pretraživanje u memoriji kako bi bilježnica radila bez Search resursa.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Alati

Alati za proizvodnju obavljaju stvarni posao nad stvarnim sustavima. Ovdje simuliramo bazu podataka narudžbi i sustav za izdavanje karata pomoću običnih Python funkcija. Dekorator `@tool` izlaže ih agentu.

Primijetite da `issue_refund` koristi `approval_mode="always_require"` za povrate iznad praga — ovo je primitivni oblik ljudske kontrole koji uvodimo kasnije.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Baza znanja o pravilima

Pitanja o pravilima ("koliki je vaš rok za povrat?") trebaju se odgovarati iz autoritativnog izvora, a ne iz memorije modela. Omotavamo malu bazu znanja kao alat za pretraživanje.

U produkciji ovo je **Azure AI Search**; ovdje nudimo pretraživanje po ključnim riječima u memoriji kako bi bilježnica radila bilo gdje, te se automatski prebacujemo na Azure AI Search kada su prisutne varijable okoline.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Memorija

Agent podrške koji zaboravi s kim razgovara loš je agent podrške. Čuvamo mali spremnik profila po kupcu i ubacujemo kratak sažetak u upute agenta. U produkciji je ovo servis memorije (vidi Lekciju 13); ovdje dict čini uzorak vidljivim.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. Model Ruta i Keširanje Odgovora

Dva mehanizma za kontrolu troškova povezana u jedinstveni upravljač zahtjevima:

- **Ruta**: jeftin heuristički klasifikator odlučuje treba li zahtjev mali ili veliki model.
- **Keširanje**: normalizirana ponovljena pitanja se poslužuju direktno iz keša bez poziva modela.

Klasifikator ovdje je namjerno jednostavan. U produkciji biste ga validirali prema prometu i mogli biste ga zamijeniti Foundryevim Model Routerom.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. Agent, Ljudsko odobrenje i Promatranje

Sada sastavljamo agenta iz gore navedenih alata i omotavamo svaki zahtjev u OpenTelemetry span. Funkcija `handle_support_request` je proizvodni upravitelj zahtjeva: cache → route → trace → run → cache.

Ljudsko odobrenje obrađuje okvir: budući da je `issue_refund` s postavkom `approval_mode="always_require"`, izvođenje se zaustavlja i prikazuje zahtjev za odobrenje koji mi eksplicitno rješavamo.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Vrata evaluacije

Ovo su vrata za izdanje iz lekcije: offline test set ocjenjuje agenta, a implementacija se nastavlja samo ako stopa prolaza premaši prag. Ocjenjivač ovdje je jednostavna provjera preklapanja ključnih riječi kako bi bilježnica bila samostalna; u produkciji biste koristili LLM kao sudca ili okvir za evaluaciju (vidi Lekciju 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Sastavljanje: Simulirano izdanje

Donja ćelija prikazuje cijelu petlju koju lekcija opisuje: pokrenite evaluacijski "gate", i samo "implementirajte" ako prođe. Ovo je obrazac koji biste koristili u CI prije nego što promocirate verziju agenta na Foundry Agent Service.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Sažetak

Sastavili ste agenta za korisničku podršku spremnog za produkciju sa svim operativnim aspektima povezanima:

- **Alati, RAG i memorija** daju agentu sposobnosti i kontekst.
- **Usmjeravanje modela i keširanje** održavaju latenciju i troškove pod kontrolom.
- **Ljudsko odobrenje** štiti visoko rizične akcije poput velikih povrata novca.
- **Vrata evaluacije** blokiraju loša izdavanja prije nego što budu lansirana.
- **Praćenje** čini svaki zahtjev promatranim.

### Izazov

Proširite ovog agenta kako bi:

1. **Podržavao više modela** — dodajte treći sloj "razumijevanja" i usmjeravajte eskalacije/žalbe na njega.
2. **Dodali vrata evaluacije** — proširite `TEST_CASES` da uključuje scenarije odobrenja povrata i potvrdite da vrata hvataju regresije.
3. **Dodali usmjeravanje svjesno troškova** — pratite procijenjeni trošak po zahtjevu (mali vs veliki vs keš) i ispišite izvještaj o troškovima nakon serije mješovitih upita.

U sljedećem poglavlju idete suprotnim putem i pokrećete agenta u potpunosti na vlastitom računalu koristeći Microsoft Foundry Local i Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
